# 최근 CLV 수준·변화 조건부 다중시점 LightGCN M2 — Dunnhumby seed 42

직전 다중시점 모형의 CLV 백분위가 사용자 안에서 거의 변하지 않았던 문제를 겨냥한 역사적 개발구간 탐색 실험입니다.

- 현재 CLV proxy: 기준시점 직전 90일의 장바구니 수 N × 평균 장바구니금액 V
- 수준 L: log(1+현재 proxy)를 train 사용자 분포에서 robust scale
- 변화 D: log(1+현재 proxy) - log(1+28일 전 proxy)를 robust scale
- 사용자 표현: $\widetilde E_{u,t}=\operatorname{NormPreserve}[E_u^{ID}\odot\{1+\rho\tanh(L_{u,t}w_L+D_{u,t}w_D)\}]$
- $\rho=0.05$ 고정, 아이템은 순수 ID 임베딩만 사용
- 학습 anchor 이력종료일: 480·508·536·564·592·620·648·676일, 각 직후 7일 신규상품 쌍
- 개발평가: 1~683일 이력, 684~690일 신규상품
- 대조군: 동일한 8개 시점·양성표본·그래프·학습순서에서 $\rho=0$
- 고정: binary graph, uniform negative sampling, plain BPR, 표본가중·새 손실 없음, 100 epoch, 하나의 optimizer

최종 test와 holdout은 만들지 않으며, seed 42 한 번으로 유의성이나 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '5be2a6f3befeeeb0f2f42c19feb3ceb4ebf25f5d'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_dynamic_level_change_multianchor import (
    configure_dynamic_level_change,
    preflight_summary,
    run_dynamic_level_change,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_dynamic_level_change(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_dynamic_clv_level_change_multianchor_historical_screen_v1'
    )
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['anchor_history_ends'] == [480, 508, 536, 564, 592, 620, 648, 676]
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['historical_clv_proxy']['rolling_window_days'] == 90
assert summary['historical_clv_proxy']['change_lag_days'] == 28
assert summary['m2']['rho'] == 0.05
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['new_loss_term'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_dynamic_level_change(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
decision = dict(result_df.attrs['decision'])
paths = dict(result_df.attrs['result_files'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))
core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('동일 8시점 protocol의 rho=0 대조군 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', decision)
print('결과 파일:', paths)